# Dreamer World Model Tutorial

This notebook demonstrates how to train and perform inference with the Dreamer world model from TorchWM.

## What is Dreamer?

Dreamer is a model-based reinforcement learning algorithm that learns a latent dynamics model from images and uses imagined rollouts to train a policy. It consists of:

- **RSSM (Recurrent State Space Model)**: Latent dynamics model combining recurrent and variational components
- **Encoder/Decoder**: For encoding images to latent space and decoding back to images
- **Actor-Critic**: Policy and value networks trained via imagined rollouts

In [ ]:
# Setup and Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# TorchWM imports
from world_models.models.dreamer import DreamerAgent, make_env
from world_models.configs.dreamer_config import DreamerConfig
from world_models.inference.operators import DreamerOperator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration

First, let's configure the Dreamer agent. The `DreamerConfig` class provides all the hyperparameters.

In [ ]:
# Create Dreamer configuration
config = DreamerConfig()

# Customize for quick demo
config.env = "cartpole-balance"  # DMC environment
config.seed = 42
config.image_size = 64
config.batch_size = 50
config.chunk_size = 64
config.total_steps = 5000  # Short training for demo
config.eval_interval = 1000
config.save_interval = 5000
config.log_interval = 100
config.grad_clip = 100
config.free_nats = 1.0
config.action_repeat = 2
config.time_limit = 1000

print("=== Dreamer Configuration ===")
print(f"Environment: {config.env}")
print(f"Image size: {config.image_size}")
print(f"Batch size: {config.batch_size}")
print(f"Total steps: {config.total_steps}")
print(f"Latent dim: {config.latent_dim}")
print(f"Hidden dim: {config.hidden_dim}")

## Training Dreamer

Training involves collecting experience from the environment, learning the world model, and updating the policy via imagined rollouts.

In [ ]:
# Create and train the Dreamer agent
print("Creating Dreamer agent...")
agent = DreamerAgent(config)

print("Starting training...")
agent.train()

print("Training complete!")

## Evaluating the Trained Model

Let's evaluate the trained model and visualize the results.

In [ ]:
# Evaluation
print("Evaluating agent...")
eval_returns = []

for episode in range(5):
    env = make_env(config)
    obs, _ = env.reset()
    obs = obs.to(torch.float32) / 255.0 - 0.5
    
    episode_return = 0.0
    done = False
    steps = 0
    max_steps = 500
    
    while not done and steps < max_steps:
        action = agent.act(obs.unsqueeze(0).to(agent.device))
        obs, reward, done, _ = env.step(action[0].cpu().numpy())
        obs = obs.to(torch.float32) / 255.0 - 0.5
        episode_return += reward
        steps += 1
    
    eval_returns.append(episode_return)
    print(f"Episode {episode + 1}: Return = {episode_return:.2f}, Steps = {steps}")
    env.close()

print(f"\nMean evaluation return: {np.mean(eval_returns):.2f} +/- {np.std(eval_returns):.2f}")

## Inference with Dreamer Operator

The `DreamerOperator` provides a convenient interface for preprocessing observations and actions for inference.

In [ ]:
# Using the DreamerOperator for preprocessing
from PIL import Image

# Create operator with your image size and action dimension
operator = DreamerOperator(image_size=64, action_dim=6)

# Example: process an image and action
dummy_image = Image.new('RGB', (64, 64), color='red')
dummy_action = [0.0, 0.5, 0.0, 0.0, 0.0, 0.0]

processed = operator.process({'image': dummy_image, 'action': dummy_action})
print("Processed observation shape:", processed['obs'].shape)
print("Processed action shape:", processed['action'].shape)
print("Observation value range:", processed['obs'].min().item(), "to", processed['obs'].max().item())

## Understanding Dreamer Components

Let's examine the key components of the Dreamer model.

In [ ]:
# Explore model architecture
print("=== Dreamer Agent Components ===")
print(f"\nRSSM (World Model):")
print(f"  - Latent dim: {agent.rssm.latent_dim}")
print(f"  - Hidden dim: {agent.rssm.hidden_dim}")
print(f"  - Embedding dim: {agent.rssm.embed_dim}")

print(f"\nEncoder:")
print(f"  - Output dim: {agent.encoder.latent_dim}")

print(f"\nDecoder:")
print(f"  - Output shape: {agent.decoder.output_shape}")

print(f"\nActor (Policy):")
print(f"  - Action dim: {agent.actor.action_dim}")

print(f"\nCritic:")
print(f"  - Latent dim: {agent.critic.latent_dim}")

## Training on Different Environments

Dreamer supports various environments including DMC, Gym, and Unity.

In [ ]:
# Example: Configure for different backends

# For DeepMind Control
dmc_config = DreamerConfig()
dmc_config.env = "walker-walk"
dmc_config.env_backend = "dmc"
dmc_config.total_steps = 1000  # Short for demo

# Create a quick test agent
dmc_agent = DreamerAgent(dmc_config)
print(f"DMC Config: env={dmc_config.env}, backend={dmc_config.env_backend}")
print(f"Action space: {dmc_agent.env.action_space}")

# Train briefly
dmc_agent.train()
print("DMC training complete!")

## Summary

In this tutorial, you learned:

1. **Configuration**: How to set up `DreamerConfig` with customizable hyperparameters
2. **Training**: How to train a DreamerAgent on DMC, Gym, or Unity environments
3. **Inference Operators**: How to use `DreamerOperator` for preprocessing
4. **Evaluation**: How to evaluate a trained model
5. **Model Components**: Understanding the RSSM, encoder, decoder, actor, and critic